# Diabetes Prediction — Exploratory Data Analysis

This notebook explores the **Pima Indians Diabetes Dataset** before model training.
The goals are:
1. Understand the class distribution and address imbalance
2. Identify and quantify missing data encoded as biological zeros
3. Characterise per-feature distributions split by outcome
4. Measure inter-feature correlations and correlations with the target
5. Detect outliers that may affect model performance
6. Summarise findings to justify the preprocessing choices in `src/data/dataset.py`

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.dataset import FEATURE_NAMES, ZERO_AS_MISSING, load_raw

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

FIGURES = '../outputs/figures'
import os; os.makedirs(FIGURES, exist_ok=True)

## 1. Load Data & Basic Statistics

In [ ]:
df_raw = pd.read_csv('../data/raw/diabetes.csv')
print(f'Dataset shape: {df_raw.shape}')
print(f'\nClass balance:\n{df_raw.Outcome.value_counts()}')
df_raw.head()

In [ ]:
df_raw.describe().T.round(2)

## 2. Class Distribution

The dataset is **imbalanced**: ~65% negative, ~35% positive. This motivates the use of SMOTE during training.

In [ ]:
counts = df_raw['Outcome'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['No Diabetes (0)', 'Diabetes (1)'], counts.values,
              color=['steelblue', 'tomato'], edgecolor='black')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            str(int(bar.get_height())), ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Count')
ax.set_title(f'Class Distribution  (imbalance ratio {counts[0]/counts[1]:.2f}:1)')
plt.tight_layout()
plt.savefig(f'{FIGURES}/class_distribution.png', dpi=150)
plt.show()

## 3. Missing Values (biologically impossible zeros)

Five features contain zeros that are clinically impossible (e.g., Glucose = 0 is physiologically impossible). These are treated as `NaN` and later imputed with the training-set median.

In [ ]:
df = load_raw('../data/raw/diabetes.csv')
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)
print('Missing % after zero-to-NaN replacement:')
print(missing_pct.to_string())

fig, ax = plt.subplots(figsize=(6, 3))
missing_pct.plot(kind='bar', color='steelblue', edgecolor='black', ax=ax)
ax.set_ylabel('Missing (%)')
ax.set_title('Missing Data by Feature')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig(f'{FIGURES}/missing_data.png', dpi=150)
plt.show()

## 4. Feature Distributions by Outcome

KDE plots reveal which features separate the two classes most clearly.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, feat in zip(axes.flatten(), FEATURE_NAMES):
    for outcome, color, label in [(0, 'steelblue', 'No Diabetes'), (1, 'tomato', 'Diabetes')]:
        subset = df[df['Outcome'] == outcome][feat].dropna()
        subset.plot(kind='kde', ax=ax, color=color, label=label, linewidth=2)
    ax.set_title(feat)
    ax.legend(fontsize=7)
plt.suptitle('Feature Distributions by Outcome', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIGURES}/feature_distributions.png', dpi=150)
plt.show()

## 5. Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df_raw.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            ax=ax, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig(f'{FIGURES}/correlation_heatmap.png', dpi=150)
plt.show()

## 6. Feature Correlations with Target

Point-biserial correlation of each feature with the binary Outcome label — a quick guide to which features the models should find most predictive.

In [ ]:
target_corr = df_raw[FEATURE_NAMES + ['Outcome']].corr()['Outcome'].drop('Outcome').sort_values(ascending=False)
print('Correlation with Outcome (Pearson):\n')
print(target_corr.round(3).to_string())

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['tomato' if v > 0 else 'steelblue' for v in target_corr]
ax.bar(target_corr.index, target_corr.values, color=colors, edgecolor='black', alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Pearson r with Outcome')
ax.set_title('Feature Correlation with Diabetes Outcome')
ax.set_xticklabels(target_corr.index, rotation=30, ha='right')
plt.tight_layout()
plt.savefig(f'{FIGURES}/target_correlation.png', dpi=150)
plt.show()

## 7. Boxplots — Outlier Detection

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, feat in zip(axes.flatten(), FEATURE_NAMES):
    df_raw.boxplot(column=feat, by='Outcome', ax=ax)
    ax.set_title(feat)
    ax.set_xlabel('Outcome (0=No Diabetes, 1=Diabetes)')
plt.suptitle('Boxplots by Outcome', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIGURES}/boxplots.png', dpi=150)
plt.show()

## 8. Pairplot — Top 4 Most Correlated Features

Visualises pairwise relationships for the four features most correlated with the outcome.

In [ ]:
top4 = target_corr.head(4).index.tolist()
print(f'Top 4 features: {top4}')
pair_df = df_raw[top4 + ['Outcome']].copy()
pair_df['Outcome'] = pair_df['Outcome'].map({0: 'No Diabetes', 1: 'Diabetes'})
g = sns.pairplot(pair_df, hue='Outcome', palette={'No Diabetes': 'steelblue', 'Diabetes': 'tomato'},
                 plot_kws={'alpha': 0.5, 's': 20}, diag_kind='kde')
g.fig.suptitle('Pairplot — Top 4 Features', y=1.02, fontsize=13)
plt.savefig(f'{FIGURES}/pairplot_top4.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Key Observations & Preprocessing Justification

| Finding | Preprocessing decision |
|---|---|
| **Class imbalance** (~65% / 35%) | SMOTE oversampling applied to training set only |
| **Biological zeros** in Glucose, BMI, Insulin, BloodPressure, SkinThickness | Replace with `NaN`, impute with training-set median |
| **Insulin** has 49% missing, **SkinThickness** 30% | Median imputation is robust; alternatives (e.g., KNN) explored but median suffices given RF's tolerance |
| **Glucose** and **BMI** have the strongest correlation with Outcome (r ≈ 0.47, 0.29) | These are consistently ranked in top-5 by both ANOVA F-test and RF importance |
| **Insulin** and **SkinThickness** show heavy right tails | Tree-based models handle this; LR benefits from StandardScaler applied upstream |
| Features are on different scales | StandardScaler applied after imputation, fit on train set only to prevent leakage |
